## Imports and Setup 

In [1]:
import  pandas as pd
import numpy as np
import json
from pandas import json_normalize
import matplotlib.pyplot as plt
pd.set_option('future.no_silent_downcasting', True)
import matplotlib.ticker as mticker
import ast
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import re
from datetime import datetime

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Plotting style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13, 'axes.labelsize': 11})
PALETTE   = sns.color_palette('muted')
FIG_BG    = '#F8F9FA'
ACCENT    = '#2B6CB0'
plt.rcParams.update({'figure.facecolor': FIG_BG, 'axes.facecolor': FIG_BG,
                     'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False})

# Primary Data Understanding

In [2]:
MASTER_SCHEMA = [

    # -------------------------
    # 1. Identification
    # -------------------------
    "url",
    "contrat",
    "type",
    "prix",
    "titre",
    "description",

    # -------------------------
    # 3. Price & Physical Features
    # -------------------------
    "surface",
    "pieces",
    "etage",

    # -------------------------
    # 4. Location
    # -------------------------
    "code_postal",
    "adresse",
    "ville",
    "gouvernerat",
    "latitude",
    "longitude",

    # -------------------------
    # 5. Temporal Information
    # -------------------------
    "date_publication",

    # -------------------------
    # 6. Property Quality
    # -------------------------
    "standing",
    "annee_constr",

    # -------------------------
    # 7. Nearby Amenities
    # -------------------------
    "ecole",
    "pharmacie",
    "hopital",
    "marche",
    "magasin",
    "bar",
    "restaurant",
    "transport",
    "caracteristiques",

    # -------------------------
    # 8. Content & Media
    # -------------------------
    "images"
]

## Monbien.tn

In [3]:
data = pd.read_json('datasets/data/monbien_tn/data.json')
df_monbien = json_normalize(data.to_dict('records'))
df_monbien.info()
df_monbien

FileNotFoundError: File datasets/data/monbien_tn/data.json does not exist

In [ ]:
#remove empty strings and common null indicators
df_monbien = df_monbien.replace(
    ["", " ", "NA", "N/A", "na", "null", "None", None],
    np.nan
)
null_table = (
    pd.DataFrame({
        "Null Count": df_monbien.isna().sum(),
        "Null Percentage (%)": df_monbien.isna().mean() * 100
    })
    .round(2)
    .sort_values(by="Null Percentage (%)", ascending=False)
)

print(null_table)


### Data Prep

In [ ]:
df_monbien_std = df_monbien.drop(columns=[col for col in df_monbien.columns if df_monbien[col].isna().all()])
df_monbien_std.info()

In [ ]:
def normalize_schema(df, mapping):
    # rename columns to master names
    df = df.rename(columns=mapping)

    # add missing columns
    for col in MASTER_SCHEMA:
        if col not in df.columns:
            df[col] = None

    # keep only master schema order
    df = df[MASTER_SCHEMA]

    return df

In [ ]:
mapping_monbien = {
    "url": "url",
    "transaction": "contrat",
    "description": "description",
    "titre": "titre",
    "pieces": "pieces",
    "prix": "prix",
    "type_bien": "type",
    "surface_m2": "surface",
    "localisation": "ville",
    "caracteristiques_csv": "caracteristiques",
    "images_csv": "images"
}
df_monbien_std = normalize_schema(df_monbien_std, mapping_monbien)
df_monbien_std.info()
df_monbien_std

In [ ]:
df_monbien_std["description"] = np.where(
    df_monbien_std["caracteristiques"].notna(),
    df_monbien_std["description"].fillna("") +
    "\nCaracteristiques: " +
    df_monbien_std["caracteristiques"],
    df_monbien_std["description"]
)

df_monbien_std["caracteristiques"] = np.nan
df_monbien_std

#### Data type conversion

In [ ]:
df_monbien_std["prix"] = (
    df_monbien_std["prix"]
    .astype(str)
    .str.replace(r"[^\d.]", "", regex=True)  # keep only numbers
)

df_monbien_std["prix"] = pd.to_numeric(df_monbien_std["prix"], errors="coerce")

df_monbien_std["surface"] = pd.to_numeric(df_monbien_std["surface"], errors="coerce")

df_monbien_std["etage"] = pd.to_numeric(df_monbien_std["etage"], errors="coerce")
df_monbien_std["annee_constr"] = pd.to_numeric(df_monbien_std["annee_constr"], errors="coerce")

df_monbien_std["pieces"] = pd.to_numeric(df_monbien_std["pieces"], errors="coerce")
df_monbien_std["pieces"] += 1

df_monbien_std["date_publication"] = pd.to_datetime(df_monbien_std["date_publication"], errors="coerce")

df_monbien_std


## Immobilier.tn

In [8]:
data = pd.read_json('datasets/data/immobilier_com_tn/data.json')
df_immobilier_tn = json_normalize(data.to_dict('records'))
df_immobilier_tn.info()
df_immobilier_tn

FileNotFoundError: File datasets/data/immobilier_com_tn/data.json does not exist

In [ ]:
df_immobilier_tn = df_immobilier_tn.replace(
    ["", " ", "NA", "N/A", "na", "null", "None", None],
    np.nan
)
null_table = (
    pd.DataFrame({
        "Null Count": df_immobilier_tn.isna().sum(),
        "Null Percentage (%)": df_immobilier_tn.isna().mean() * 100
    })
    .round(2)
    .sort_values(by="Null Percentage (%)", ascending=False)
)

print(null_table)

### Data Prep

In [ ]:
df_immobilier_tn_std = df_immobilier_tn.drop(columns=[col for col in df_immobilier_tn.columns if df_immobilier_tn[col].isna().all()])
df_immobilier_tn_std.info()

In [ ]:
def normalize_schema(df, mapping):
    # rename columns to master names
    df = df.rename(columns=mapping)

    # add missing columns
    for col in MASTER_SCHEMA:
        if col not in df.columns:
            df[col] = None

    # keep only master schema order
    df = df[MASTER_SCHEMA]

    return df

In [ ]:
mapping_immobilier_tn = {
    "url": "url",
    "transaction": "contrat",
    "description": "description",
    "titre": "titre",
    "pieces": "pieces",
    "prix": "prix",
    "adresse": "adresse",
    "type_bien": "type",
    "surface_m2": "surface",
    "localisation": "ville",
    "caracteristiques_csv": "caracteristiques",
    "images_csv": "images"
}
df_immobilier_tn_std = normalize_schema(df_immobilier_tn_std, mapping_immobilier_tn)
df_immobilier_tn_std.info()
df_immobilier_tn_std

In [ ]:
df_immobilier_tn_std["description"] = np.where(
    df_immobilier_tn_std["caracteristiques"].notna(),
    df_immobilier_tn_std["description"].fillna("") +
    "\nCaracteristiques: " +
    df_immobilier_tn_std["caracteristiques"],
    df_immobilier_tn_std["description"]
)

df_immobilier_tn_std["caracteristiques"] = np.nan
df_immobilier_tn_std

#### Data type conversion

In [ ]:
df_immobilier_tn_std["prix"] = (
    df_immobilier_tn_std["prix"]
    .astype(str)
    .str.replace(r"[^\d.]", "", regex=True)  # keep only numbers
)

df_immobilier_tn_std["prix"] = pd.to_numeric(df_immobilier_tn_std["prix"], errors="coerce")

df_immobilier_tn_std["surface"] = pd.to_numeric(df_immobilier_tn_std["surface"], errors="coerce")

df_immobilier_tn_std["etage"] = pd.to_numeric(df_immobilier_tn_std["etage"], errors="coerce")
df_immobilier_tn_std["annee_constr"] = pd.to_numeric(df_immobilier_tn_std["annee_constr"], errors="coerce")

df_immobilier_tn_std["pieces"] = pd.to_numeric(df_immobilier_tn_std["pieces"], errors="coerce")
df_immobilier_tn_std["pieces"] += 1

df_immobilier_tn_std["date_publication"] = pd.to_datetime(df_immobilier_tn_std["date_publication"], errors="coerce")

df_immobilier_tn_std


## Annonces Immobilieres

In [ ]:
data = pd.read_json('datasets/annonces_immobilieres/data.json')
df_annonces_immo = json_normalize(data.to_dict('records'))
df_annonces_immo.info()
df_annonces_immo

In [ ]:
df_annonces_immo = df_annonces_immo.replace(
    ["", " ", "NA", "N/A", "na", "null", "None", None],
    np.nan
)
null_table = (
    pd.DataFrame({
        "Null Count": df_annonces_immo.isna().sum(),
        "Null Percentage (%)": df_annonces_immo.isna().mean() * 100
    })
    .round(2)
    .sort_values(by="Null Percentage (%)", ascending=False)
)

print(null_table)

### Data Prep

In [ ]:
df_annonces_immo_std = df_annonces_immo.drop(columns=[col for col in df_annonces_immo.columns if df_annonces_immo[col].isna().all()])
df_annonces_immo_std.info()

In [ ]:
def normalize_schema(df, mapping):
    # rename columns to master names
    df = df.rename(columns=mapping)

    # add missing columns
    for col in MASTER_SCHEMA:
        if col not in df.columns:
            df[col] = None

    # keep only master schema order
    df = df[MASTER_SCHEMA]

    return df

In [ ]:
df_annonces_immo_std["caracteristiques_csv"]+= "\n details : " + df_annonces_immo_std["details_csv"]

print (df_annonces_immo_std["caracteristiques_csv"][0])
print ("==================================================")
print (df_annonces_immo_std["details_csv"][0])


In [ ]:

mapping_annonces_immo = {
    "url": "url",
    "transaction": "contrat",
    "description": "description",
    "titre": "titre",
    "pieces": "pieces",
    "prix": "prix",
    "adresse": "adresse",
    "type_bien": "type",
    "surface_m2": "surface",
    "localisation": "ville",
    "caracteristiques_csv": "caracteristiques",
    "images_csv": "images"
}
df_annonces_immo_std = normalize_schema(df_annonces_immo_std, mapping_annonces_immo)
df_annonces_immo_std.info()
df_annonces_immo_std

In [ ]:
df_annonces_immo_std["description"] = np.where(
    df_annonces_immo_std["caracteristiques"].notna(),
    df_annonces_immo_std["description"].fillna("") +
    "\nCaracteristiques: " +
    df_annonces_immo_std["caracteristiques"],
    df_annonces_immo_std["description"]
)

df_annonces_immo_std["caracteristiques"] = np.nan
df_annonces_immo_std

#### Data type conversion

In [ ]:
df_annonces_immo_std["prix"] = (
    df_annonces_immo_std["prix"]
    .astype(str)
    .str.replace(r"[^\d.]", "", regex=True)  # keep only numbers
)

df_annonces_immo_std["prix"] = pd.to_numeric(df_annonces_immo_std["prix"], errors="coerce")

df_annonces_immo_std["surface"] = pd.to_numeric(df_annonces_immo_std["surface"], errors="coerce")

df_annonces_immo_std["etage"] = pd.to_numeric(df_annonces_immo_std["etage"], errors="coerce")
df_annonces_immo_std["annee_constr"] = pd.to_numeric(df_annonces_immo_std["annee_constr"], errors="coerce")

df_immobilier_tn_std["pieces"] = pd.to_numeric(df_immobilier_tn_std["pieces"], errors="coerce")

df_immobilier_tn_std["date_publication"] = pd.to_datetime(df_immobilier_tn_std["date_publication"], errors="coerce")

df_immobilier_tn_std


## Tunisie Annonce

In [ ]:
data = pd.read_json('datasets/tunisie_annonce/data.json')
df_tunisie_annonce = json_normalize(data.to_dict('records'))
df_tunisie_annonce.info()
df_tunisie_annonce

In [ ]:
df_tunisie_annonce = df_tunisie_annonce.replace(
    ["", " ", "NA", "N/A", "na", "null", "None", None],
    np.nan
)
null_table = (
    pd.DataFrame({
        "Null Count": df_tunisie_annonce.isna().sum(),
        "Null Percentage (%)": df_tunisie_annonce.isna().mean() * 100
    })
    .round(2)
    .sort_values(by="Null Percentage (%)", ascending=False)
)

print(null_table)

### Data Prep

In [ ]:
df_tunisie_annonce_std = df_tunisie_annonce.drop(columns=[col for col in df_tunisie_annonce.columns if df_tunisie_annonce[col].isna().all()])
df_tunisie_annonce_std.info()

In [ ]:
def normalize_schema(df, mapping):
    # rename columns to master names
    df = df.rename(columns=mapping)

    # add missing columns
    for col in MASTER_SCHEMA:
        if col not in df.columns:
            df[col] = None

    # keep only master schema order
    df = df[MASTER_SCHEMA]

    return df

In [ ]:
mapping_tunisie_annonce = {
    "title_full": "titre",
    "description_full": "description",
    "locality_detail": "adresse",
    "city": "ville",
    "region": "gouvernerat",
    "detail_url": "url",
    "price_tnd": "prix",
    "surface_m2": "surface",
    "property_subtype" : "type",
    "date_posted": "date_publication",
    "category_type": "contrat",
}
df_tunisie_annonce_std = normalize_schema(df_tunisie_annonce_std, mapping_tunisie_annonce)
df_tunisie_annonce_std.info()
df_tunisie_annonce_std

In [ ]:
df_tunisie_annonce_std["pieces"] = df_tunisie_annonce_std["type"].str.extract(r"(\d+)")  # extract digits

In [ ]:
df_tunisie_annonce_std["type"] = df_tunisie_annonce_std["type"].str.split().str[0]

In [ ]:
df_tunisie_annonce_std

#### Data type conversion

In [ ]:
df_tunisie_annonce_std["prix"] = (
    df_tunisie_annonce_std["prix"]
    .astype(str)
    .str.replace(r"[^\d.]", "", regex=True)  # keep only numbers
)

df_tunisie_annonce_std["prix"] = pd.to_numeric(df_tunisie_annonce_std["prix"], errors="coerce")

df_tunisie_annonce_std["surface"] = pd.to_numeric(df_tunisie_annonce_std["surface"], errors="coerce")

df_tunisie_annonce_std["etage"] = pd.to_numeric(df_tunisie_annonce_std["etage"], errors="coerce")
df_tunisie_annonce_std["annee_constr"] = pd.to_numeric(df_tunisie_annonce_std["annee_constr"], errors="coerce")

df_tunisie_annonce_std["pieces"] = pd.to_numeric(df_tunisie_annonce_std["pieces"], errors="coerce")

df_tunisie_annonce_std["date_publication"] = pd.to_datetime(df_tunisie_annonce_std["date_publication"], errors="coerce")

df_tunisie_annonce_std


## Bigdatis

In [ ]:
data = pd.read_json('datasets/Bigdatis/data1.json')
df_bigdatis = json_normalize(data.to_dict('records'))
df_bigdatis.info()
df_bigdatis

In [ ]:
df_bigdatis = df_bigdatis.replace(
    ["", " ", "NA", "N/A", "na", "null", "None", None],
    np.nan
)
null_table = (
    pd.DataFrame({
        "Null Count": df_bigdatis.isna().sum(),
        "Null Percentage (%)": df_bigdatis.isna().mean() * 100
    })
    .round(2)
    .sort_values(by="Null Percentage (%)", ascending=False)
)

print(null_table)

### Data Prep

In [ ]:
df_bigdatis_std = df_bigdatis.drop(columns=[col for col in df_bigdatis.columns if df_bigdatis[col].isna().all()])
df_bigdatis_std.info()

In [ ]:
def normalize_schema(df, mapping):
    # rename columns to master names
    df = df.rename(columns=mapping)

    # add missing columns
    for col in MASTER_SCHEMA:
        if col not in df.columns:
            df[col] = None

    # keep only master schema order
    df = df[MASTER_SCHEMA]

    return df

In [ ]:
df_bigdatis_std.drop(columns=["images"], inplace=True)

mapping_bigdatis= {
    "url": "url",
    "properties.transactionType": "contrat",
    "description": "description",
    "title": "titre",
    "properties.typology": "pieces",
    "price": "prix",
    "locationId": "adresse",
    "properties.propertyType": "type",
    "area": "surface",
    "flags": "caracteristiques",
    "imageUrls": "images",
    "modifiedAt": "date_publication"
}
df_bigdatis_std = normalize_schema(df_bigdatis_std, mapping_bigdatis)
df_bigdatis_std.info()
df_bigdatis_std

#### Data type conversion

In [ ]:
df_bigdatis_std["prix"] = (
    df_bigdatis_std["prix"]
    .astype(str)
    .str.replace(r"[^\d.]", "", regex=True)  # keep only numbers
)

df_bigdatis_std["prix"] = pd.to_numeric(df_bigdatis_std["prix"], errors="coerce")

df_bigdatis_std["surface"] = pd.to_numeric(df_bigdatis_std["surface"], errors="coerce")

df_bigdatis_std["etage"] = pd.to_numeric(df_bigdatis_std["etage"], errors="coerce")
df_bigdatis_std["annee_constr"] = pd.to_numeric(df_bigdatis_std["annee_constr"], errors="coerce")

df_bigdatis_std["pieces"] = pd.to_numeric(df_bigdatis_std["pieces"], errors="coerce")

df_bigdatis_std["date_publication"] = pd.to_datetime(df_bigdatis_std["date_publication"], errors="coerce")

df_bigdatis_std


## Tecnocasa

In [ ]:
data = pd.read_json('datasets/tecnocasa/data.json')
df_tecnocasa = json_normalize(data.to_dict('records'))
df_tecnocasa.info()
df_tecnocasa

In [ ]:
df_tecnocasa = df_tecnocasa.replace(
    ["", " ", "NA", "N/A", "na", "null", "None", None],
    np.nan
)
null_table = (
    pd.DataFrame({
        "Null Count": df_tecnocasa.isna().sum(),
        "Null Percentage (%)": df_tecnocasa.isna().mean() * 100
    })
    .round(2)
    .sort_values(by="Null Percentage (%)", ascending=False)
)

print(null_table)

### Data Prep

In [ ]:
df_tecnocasa_std = df_tecnocasa.drop(columns=[col for col in df_tecnocasa.columns if df_tecnocasa[col].isna().all()])
df_tecnocasa_std.info()

**Combine caracteristics into one column**

In [ ]:
columns = [
    "services.group.2.nome-macro",
    "services.group.0.nome-macro",
    "services.group.6.nome-macro",
    "services.group.1.nome-macro",
    "services.group.4.nome-macro",
    "services.group.5.nome-macro",
    "services.group.7.nome-macro",
    "services.group.8.nome-macro",
    "services.group.9.nome-macro",
    "services.group.10.nome-macro",
    "services.group.11.nome-macro",
]

# Combine all 3 columns into one list per row
def combine_services(row):
    values = []
    for col in columns:
        if col in df_tecnocasa_std.columns and pd.notna(row[col]):
            values.extend(str(row[col]).split(","))
    return [v.strip() for v in values if v.strip() != ""]

df_tecnocasa_std["caracteristiques"] = df_tecnocasa_std.apply(combine_services, axis=1)
df_tecnocasa_std["caracteristiques"].value_counts()

In [ ]:
def add_features_to_caracteristiques(row):
    # Ensure caracteristiques is a list
    if isinstance(row['caracteristiques'], str):
        try:
            features_list = ast.literal_eval(row['caracteristiques'])
        except:
            features_list = []
    elif isinstance(row['caracteristiques'], list):
        features_list = row['caracteristiques']
    else:
        features_list = []

    if row.get('afeatures.air_conditioning') not in [np.nan, "none", "na", "null", "None",'non',"non clim","no clim","non climatisé","no"] :
        features_list.append("clim")

    if row.get('features.elevator') not in [np.nan, "none", "na", "null", "None","non","no","No","Non"]:
        features_list.append("ascenseur")

    if row.get('features.heating') not in [np.nan, "none", "na", "null", "None","non","no","No","Non"]:
        features_list.append("chauffage")

    if row.get('features.garden') not in [np.nan, "none", "na", "null", "None","non","no","No","Non"]:
        features_list.append("jardin")

    return list(set(features_list))  # remove duplicates if any

df_tecnocasa_std['caracteristiques'] = df_tecnocasa_std.apply(add_features_to_caracteristiques, axis=1)

df_tecnocasa_std['caracteristiques'].value_counts()

In [ ]:
def normalize_schema(df, mapping):
    # rename columns to master names
    df = df.rename(columns=mapping)

    # add missing columns
    for col in MASTER_SCHEMA:
        if col not in df.columns:
            df[col] = None

    # keep only master schema order
    df = df[MASTER_SCHEMA]

    return df

In [ ]:
df_tecnocasa_std.drop(columns=["surface"], inplace=True)

mapping_tecnocasa= {
    "detail_url": "url",
    "transaction": "contrat",
    "description": "description",
    "title": "titre",
    "rooms": "pieces",
    "numeric_price": "prix",
    "latitude": "latitude",
    "province.slug":"gouvernerat",
    "longitude": "longitude",
    "district.slug": "ville",
    "address": "adresse",
    "type.slug": "type",
    "features.floor": "etage",
    "features.category": "standing",
    "features.build_year": "annee_constr",
    "numeric_surface": "surface",
    "last_published_at": "date_publication",
    "contract.title": "contrat",
    "points_of_interest.school": "ecole",
    "points_of_interest.pharmacy": "pharmacie",
    "points_of_interest.hospital": "hopital",
    "points_of_interest.market": "marche",
    "points_of_interest.shop": "magasin",
    "points_of_interest.bar": "bar",
    "points_of_interest.restaurant": "restaurant",
    "points_of_interest.public_transport": "transport"
}
df_tecnocasa_std = normalize_schema(df_tecnocasa_std, mapping_tecnocasa)
df_tecnocasa_std.info()
df_tecnocasa_std

In [ ]:
print(df_tecnocasa_std['pieces'].value_counts())
if 'pieces' in df_tecnocasa_std.columns:
    df_tecnocasa_std['pieces'] = df_tecnocasa_std['pieces'].astype(str).str.extract('(\d+)').astype('Int64')
print(df_tecnocasa_std['pieces'].unique())

if 'etage' in df_tecnocasa_std.columns:
    manual_mapping = {
    'RDC': 0,
    'Terre': 0,
    'Sous-sol': -1,
    'Sot': -1,
    'Bas': 1,
    'Moyen': 2,
    'Elevé': 3,
    'Dernier': np.nan,  # optional: choose max floor or leave NaN
    'Grenier': np.nan   # optional: choose a value
}

def process_floor(val):
    val_str = str(val).strip()
    
    # If empty, return NaN
    if val_str == '':
        return np.nan
    
    # Try extracting first number
    num = pd.Series([int(x) for x in val_str if x.isdigit()])
    if not num.empty:
        return int(num.iloc[0])
    
    # If no number, check mapping
    return manual_mapping.get(val_str, np.nan)  # default NaN if unknown

# Apply function
df_tecnocasa_std['etage'] = df_tecnocasa_std['etage'].apply(process_floor).astype('Int64')
print(df_tecnocasa_std['etage'].unique())

#### Data type conversion

In [ ]:
df_tecnocasa_std["prix"] = (
    df_tecnocasa_std["prix"]
    .astype(str)
    .str.replace(r"[^\d.]", "", regex=True)  # keep only numbers
)

df_tecnocasa_std["prix"] = pd.to_numeric(df_tecnocasa_std["prix"], errors="coerce")

df_tecnocasa_std["surface"] = pd.to_numeric(df_tecnocasa_std["surface"], errors="coerce")

df_tecnocasa_std["etage"] = pd.to_numeric(df_tecnocasa_std["etage"], errors="coerce")
df_tecnocasa_std["annee_constr"] = pd.to_numeric(df_tecnocasa_std["annee_constr"], errors="coerce")



df_tecnocasa_std["date_publication"] = pd.to_datetime(df_tecnocasa_std["date_publication"], errors="coerce")

df_tecnocasa_std


## Merging datasets 

In [ ]:
dataset_full_raw = pd.concat([df_monbien_std, df_tunisie_annonce_std, 
df_annonces_immo_std, df_immobilier_tn_std, df_tecnocasa_std, df_bigdatis_std], ignore_index=True)
dataset_full_raw.info()

**Unify the categories in categorical data**

In [ ]:
dataset_full_raw["type"].unique()

In [ ]:
TYPE_MAPPING = {
    # ---------------- Apartments
    "appartement": "appartement",
    "app.": "appartement",
    "flat": "appartement",
    "villaFloor": "appartement",

    # ---------------- Houses / Villas
    "villa": "maison",
    "house": "maison",
    "maisons": "maison",
    "duplex": "maison",
    "triplex": "maison",

    # ---------------- Land
    "terrain": "terrain",
    "ferme": "terrain",

    # ---------------- Commercial
    "local commercial": "local commercial",
    "commercialpremise": "local commercial",
    "fond": "local commercial",
    "atelier": "local commercial",
    "surfaces": "local commercial",
    "entrepôt": "local commercial",

    # ---------------- Office
    "bureau": "bureau",
    "office": "bureau",

    # ---------------- large assets / other
    "immeuble": "autre",
    "autre": "autre",
    "gérance": "autre",
}

In [ ]:
dataset_full_raw["type"] = (
    dataset_full_raw["type"]
    .astype("string")
    .str.strip()
    .str.lower()
)
dataset_full_raw["type"] = (
    dataset_full_raw["type"]
    .map(TYPE_MAPPING)
    .fillna("Autre")
)
dataset_full_raw["type"].unique()

In [ ]:
dataset_full_raw["contrat"].unique()

In [ ]:
CONTRAT_MAPPING = {

    # -------- Sale
    "vente": "vente",
    
    "terrain": "vente",  # usually land sale listings

    # -------- Rental
    "location": "location",
    "rental": "location",

    # -------- Vacation rental
    "location vacances": "location vacances",

    # -------- Shared housing
    "partage": "partage",

    # -------- Commercial category (site classification)
    "bureaux & commerces": "commercial",
    
    "achat": "achat",
}

In [ ]:
dataset_full_raw["contrat"] = (
    dataset_full_raw["contrat"]
    .astype("string")
    .str.strip()
    .str.lower()
)
dataset_full_raw["contrat"] = (
    dataset_full_raw["contrat"]
    .map(CONTRAT_MAPPING)
    .fillna("Autre")
)

dataset_full_raw["contrat"].unique()

In [ ]:
dataset_full_raw["pieces"].unique()

In [ ]:
dataset_full_raw["etage"].unique()

In [ ]:
dataset_full_raw["standing"].unique()

In [ ]:
dataset_full_raw["adresse"].unique()  ## needs to be processed further 

In [ ]:
dataset_full_raw["ville"].unique()

In [ ]:
dataset_full_raw["gouvernerat"].unique()

In [ ]:
TUNISIA_GOUVERNORATS = [
    "tunis","ariana","ben arous","manouba",
    "nabeul","zaghouan","bizerte","beja","jendouba","le kef","siliana",
    "sousse","monastir","mahdia","sfax","kairouan","kasserine","sidi bouzid",
    "gabes","medenine","tataouine","gafsa","tozeur","kebili"
]
GOUVERNORAT_MAPPING = {

    # ---- Grand Tunis
    "tunis": "tunis",
    "ariana": "ariana",
    "ben arous": "ben arous",
    "manouba": "manouba",
    "grand-tunis": "tunis",

    # ---- North / Cap Bon
    "nabeul": "nabeul",
    "cap-bon": "nabeul",
    "bizerte": "bizerte",

    # ---- Northwest
    "beja": "beja",
    "jendouba": "jendouba",
    "le kef": "le kef",
    "siliana": "siliana",

    # ---- Sahel
    "sousse": "sousse",
    "monastir": "monastir",
    "mahdia": "mahdia",

    # ---- Center
    "sfax": "sfax",
    "kairouan": "kairouan",
    "kasserine": "kasserine",
    "sidi bouzid": "sidi bouzid",

    # ---- South
    "gabes": "gabes",
    "medenine": "medenine",
    "tataouine": "tataouine",
    "gafsa": "gafsa",
    "tozeur": "tozeur",
    "kebili": "kebili",
}

In [ ]:
dataset_full_raw["gouvernerat"] = (
    dataset_full_raw["gouvernerat"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map(GOUVERNORAT_MAPPING)
)

In [ ]:
dataset_full_raw[["titre", "description", "caracteristiques"]] = (
    dataset_full_raw[["titre", "description", "caracteristiques"]]
    .astype("string")
)

# Secondary Data Understanding

## 🏠 Tunisian Real Estate — Data Understanding & Preparation
> **Dataset:** `dataset_full_raw` — 51,939 listings · 28 columns · scraped from Tunisian property portals

## 2. Basic Overview <a id='2'></a>
> Quick first look: shape, dtypes, memory, and duplicates.

In [ ]:

df = dataset_full_raw.copy()

df.applymap(type).eq(list).any()
cols_no_lists = [
    c for c in df.columns
    if not df[c].apply(lambda x: isinstance(x, list)).any()
]

subset_cols = ["adresse", "surface", "type"]

# mark rows with same address + type + surface
potential_dups = df[df.duplicated(subset=subset_cols, keep=False)]


print('='*55)
print(f'  Rows       : {df.shape[0]:,}')
print(f'  Columns    : {df.shape[1]}')
print(f'  Memory     : {df.memory_usage(deep=True).sum()/1e6:.1f} MB')
print(f"  duplicates  : {potential_dups.shape[0]:,} ({potential_dups.shape[0]/df.shape[0]*100:.2f}%)")
print('='*55)

potential_dups.sort_values(subset_cols)




In [ ]:
# Column-level overview
overview = pd.DataFrame({
    'dtype'      : df.dtypes,
    'non_null'   : df.notna().sum(),
    'null'       : df.isna().sum(),
    'null_%'     : (df.isna().mean()*100).round(1),
    'sample_val' : [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns]
})
overview


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 7), facecolor=FIG_BG)
fig.suptitle('📋 Data Quality Summary', fontsize=20, fontweight='bold', y=1.02)

# ── 1. KPI Cards ──────────────────────────────────────────────────────────────
ax = axes[0]
ax.set_facecolor(FIG_BG)
ax.axis('off')

total_rows    = dataset_full_raw.shape[0]
total_cols    = dataset_full_raw.shape[1]
empty_rows    = dataset_full_raw.isna().all(axis=1).sum()
total_missing = dataset_full_raw.isna().sum().sum()
missing_pct   = total_missing / (total_rows * total_cols) * 100

kpis = [
    ('🗂️  Total Rows',    f'{total_rows}',        '#2B6CB0'),
    ('📊  Total Columns', f'{total_cols}',            '#2B6CB0'),
    ('❌  Empty Rows',    f'{empty_rows}',          '#C53030'),
    ('📉  Total Missing', f'{total_missing}',       '#C53030'),
    ('📈  Missing %',     f'{missing_pct:.1f}%',    '#C53030'),
]

for i, (label, value, color) in enumerate(kpis):
    y = 0.85 - i * 0.22
    ax.add_patch(plt.Rectangle((0.05, y - 0.08), 0.9, 0.18,
                                transform=ax.transAxes, facecolor='white',
                                edgecolor=color, linewidth=2, zorder=2,
                                clip_on=False,
                                **{'radius': 0.02} if False else {}))
    ax.text(0.5, y + 0.04, value, transform=ax.transAxes,
            ha='center', va='center', fontsize=22, fontweight='bold',
            color=color)
    ax.text(0.5, y - 0.03, label, transform=ax.transAxes,
            ha='center', va='center', fontsize=10, color='#4A5568')

ax.set_title('Key Metrics', fontweight='bold', fontsize=13, pad=12)

# ── 2. Null Severity Donut ────────────────────────────────────────────────────
ax = axes[1]
ax.set_facecolor(FIG_BG)

severity = {
    '🟢 Good\n(<20% null)':    0,
    '🟡 Medium\n(20–50%)':     0,
    '🔴 Poor\n(>50% null)':    0,
}
good, medium, poor = [], [], []
for c in dataset_full_raw.columns:
    pct = dataset_full_raw[c].isna().mean() * 100
    if pct < 20:
        good.append(c);   severity['🟢 Good\n(<20% null)']   += 1
    elif pct < 50:
        medium.append(c); severity['🟡 Medium\n(20–50%)']    += 1
    else:
        poor.append(c);   severity['🔴 Poor\n(>50% null)']   += 1

sizes  = list(severity.values())
labels = list(severity.keys())
colors = ['#38A169', '#DD6B20', '#C53030']
explode = [0.03] * 3

wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=colors, explode=explode,
    autopct=lambda p: f'{p:.1f}%\n({int(round(p*total_cols/100))} cols)',
    startangle=90, pctdistance=0.75,
    wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2),
    textprops={'fontsize': 9}
)
for at in autotexts:
    at.set_fontsize(8.5)
    at.set_fontweight('bold')

ax.set_title('Columns by Null Severity', fontweight='bold', fontsize=13, pad=12)

# ── 3. Per-Column Missing % Bar ───────────────────────────────────────────────
ax = axes[2]
ax.set_facecolor(FIG_BG)

miss = (dataset_full_raw.isna().mean() * 100).sort_values(ascending=True)
bar_colors = ['#38A169' if v < 20 else '#DD6B20' if v < 50 else '#C53030' for v in miss]

bars = ax.barh(miss.index, miss.values, color=bar_colors,
               edgecolor='white', linewidth=0.5, height=0.7)

ax.axvline(20, color='#38A169', linestyle='--', linewidth=1.2, alpha=0.7, label='20% threshold')
ax.axvline(50, color='#DD6B20', linestyle='--', linewidth=1.2, alpha=0.7, label='50% threshold')

for bar, val in zip(bars, miss.values):
    if val > 2:
        ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=7.5, color='#2D3748')

ax.set_xlabel('Missing %', fontsize=10)
ax.set_title('Missing % per Column', fontweight='bold', fontsize=13, pad=12)
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend(fontsize=8, loc='lower right')
ax.set_xlim(0, 110)

# ── Legend patch for bar chart ────────────────────────────────────────────────
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#38A169', label='Good (<20%)'),
                   Patch(facecolor='#DD6B20', label='Medium (20–50%)'),
                   Patch(facecolor='#C53030', label='Poor (>50%)')]
ax.legend(handles=legend_elements, fontsize=8, loc='lower right')

plt.tight_layout()
plt.savefig('data_quality_summary.png', dpi=150, bbox_inches='tight',
            facecolor=FIG_BG)
plt.show()

## 3. Missing Value Analysis <a id='3'></a>
- 🔴 **>70% missing** → consider dropping  
- 🟠 **30–70% missing** → impute carefully or use as flag  
- 🟢 **<30% missing** → standard imputation


In [ ]:
missing = pd.DataFrame({
    'missing_count' : df.isnull().sum(),
    'missing_pct'   : (df.isnull().mean()*100).round(1),
}).sort_values('missing_pct', ascending=False)
missing = missing[missing['missing_count'] > 0]

def highlight(val):
    if val > 70:
        return 'background-color: #c0392b; color: white'  # dark red
    elif val > 30:
        return 'background-color: #d35400; color: white'  # dark orange
    else:
        return 'background-color: #27ae60; color: white'  # dark green

missing.style.applymap(highlight, subset=['missing_pct'])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Left — bar chart
colors = ['#e74c3c' if v > 70 else '#e67e22' if v > 30 else '#2ecc71'
          for v in missing['missing_pct']]
axes[0].barh(missing.index, missing['missing_pct'], color=colors)
axes[0].axvline(30, color='orange', linestyle='--', alpha=0.8, label='30%')
axes[0].axvline(70, color='red',    linestyle='--', alpha=0.8, label='70%')
axes[0].set_xlabel('Missing %')
axes[0].set_title('Missing Values per Column')
axes[0].legend()
axes[0].invert_yaxis()

# Right — heatmap pattern (500-row sample)
sample = df.sample(min(500, len(df)), random_state=42)
sns.heatmap(sample[missing.index].isnull(), cbar=False,
            yticklabels=False, cmap='YlOrRd', ax=axes[1])
axes[1].set_title('Missingness Pattern (500-row sample)')
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('Missing Value Analysis', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 4. Numeric Distributions & Outliers <a id='4'></a>
Key numeric features: `prix`, `surface`, `pieces`, `etage`, `annee_constr`

In [ ]:
numeric_cols = ['prix', 'surface', 'pieces', 'etage', 'annee_constr']
df[numeric_cols].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T.style.format('{:.2f}')


In [ ]:
# IQR Outlier Detection
print('-'*55)
for col in ['prix', 'surface', 'pieces']:
    s = df[col].dropna()
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR = Q3 - Q1
    lo, hi = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    n_out = ((s < lo) | (s > hi)).sum()
    print(f'{col:<16} {n_out:>9,} {n_out/len(s)*100:>6.1f}%  [{lo:,.0f} — {hi:,.0f}]')


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
plot_cols = ['prix', 'surface', 'pieces', 'etage', 'annee_constr']
colors     = ['#3498db', '#2ecc71', '#e67e22', '#9b59b6', '#1abc9c']

for ax, col, color in zip(axes.flat, plot_cols, colors):
    data = df[col].dropna()
    clipped = data.clip(data.quantile(0.01), data.quantile(0.99))
    ax.hist(clipped, bins=50, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(data.median(), color='red',    ls='--', lw=1.5, label=f'Median {data.median():.0f}')
    ax.axvline(data.mean(),   color='black',  ls='--', lw=1.5, label=f'Mean {data.mean():.0f}')
    ax.set_title(f'{col}  (n={len(data):,})')
    ax.set_xlabel(col)
    ax.legend(fontsize=8)

# 6th plot — prix vs surface scatter
ax = axes[1, 2]
mask = df['prix'].notna() & df['surface'].notna()
ax.scatter(
    df.loc[mask, 'surface'].clip(0, df['surface'].quantile(0.98)),
    df.loc[mask, 'prix'].clip(0, df['prix'].quantile(0.98)),
    alpha=0.15, s=5, color='#e74c3c'
)
ax.set_xlabel('Surface (m²)')
ax.set_ylabel('Prix (TND)')
ax.set_title('Prix vs Surface')

plt.suptitle('Numeric Feature Distributions (clipped 1–99th pct)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 5. Categorical Analysis <a id='5'></a>
Key categorical features: `contrat`, `type`, `gouvernerat`, `standing`

In [ ]:
cat_cols = ['contrat', 'type', 'gouvernerat', 'standing']

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
for ax, col in zip(axes.flat, cat_cols):
    vc = df[col].value_counts(dropna=False).head(12)
    bars = ax.barh(vc.index.astype(str), vc.values,
                   color=plt.cm.tab20.colors[:len(vc)])
    ax.bar_label(bars, fmt='%,.0f', padding=3, fontsize=8)
    ax.set_title(f'{col}  (unique: {df[col].nunique()})')
    ax.set_xlabel('Count')
    ax.invert_yaxis()

plt.suptitle('Categorical Feature Distributions (Top 12)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Price distribution by contrat type
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, col in zip(axes, ['contrat', 'type']):
    top_cats = df[col].value_counts().head(6).index
    data = [df.loc[df[col]==cat, 'prix'].dropna().clip(0, df['prix'].quantile(0.95))
            for cat in top_cats]
    ax.boxplot(data, labels=top_cats, patch_artist=True,
               boxprops=dict(facecolor='#3498db', alpha=0.6))
    ax.set_title(f'Prix distribution by {col}')
    ax.set_ylabel('Prix (TND)')
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()


## 6. Temporal Analysis <a id='6'></a>

In [ ]:
df['date_publication'] = pd.to_datetime(df['date_publication'], errors='coerce')

print(f"Date range   : {df['date_publication'].min().date()} → {df['date_publication'].max().date()}")
print(f"Coverage     : {df['date_publication'].notna().sum():,} / {len(df):,} rows "
      f"({df['date_publication'].notna().mean()*100:.1f}%)")

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Monthly trend
monthly = df.groupby(df['date_publication'].dt.to_period('M')).size()
monthly.plot(ax=axes[0], color='#2ecc71', linewidth=2)
axes[0].fill_between(range(len(monthly)), monthly.values, alpha=0.2, color='#2ecc71')
axes[0].set_title('Listings per Month')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Count')

# By year
yr = df['date_publication'].dt.year.value_counts().sort_index()
axes[1].bar(yr.index.astype(str), yr.values, color='#3498db', edgecolor='white')
axes[1].set_title('Listings per Year')
axes[1].tick_params(axis='x', rotation=45)

# By month of year
mon = df['date_publication'].dt.month.dropna().astype(int).value_counts().sort_index()
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
axes[2].bar([month_names[m-1] for m in mon.index], mon.values, color='#9b59b6', edgecolor='white')
axes[2].set_title('Listings by Month of Year (Seasonality)')
axes[2].tick_params(axis='x', rotation=45)

plt.suptitle('Temporal Analysis', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 7. Geographic Analysis <a id='7'></a>

In [ ]:
print(f"Listings with GPS : {df['latitude'].notna().sum():,} ({df['latitude'].notna().mean()*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

top_govs = df['gouvernerat'].value_counts().head(15).index

# Count by gouvernerat
cnt = df['gouvernerat'].value_counts().head(15)
axes[0].barh(cnt.index, cnt.values, color='#3498db')
axes[0].set_title('Listing Count by Gouvernerat (Top 15)')
axes[0].set_xlabel('Number of Listings')
axes[0].invert_yaxis()

# Median price by gouvernerat
med_price = (df[df['gouvernerat'].isin(top_govs)]
             .groupby('gouvernerat')['prix']
             .median()
             .sort_values(ascending=False))
axes[1].barh(med_price.index, med_price.values, color='#e67e22')
axes[1].set_title('Median Prix by Gouvernerat (TND)')
axes[1].set_xlabel('Median Prix (TND)')
axes[1].invert_yaxis()

plt.suptitle('Geographic Analysis', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# GPS scatter plot
gps = df.dropna(subset=['latitude','longitude'])
# Filter to Tunisia bounding box
gps = gps[(gps['latitude'].between(30, 38)) & (gps['longitude'].between(7, 12))]

fig, ax = plt.subplots(figsize=(5, 7))
scatter = ax.scatter(gps['longitude'], gps['latitude'],
                     c=np.log1p(gps['prix']), cmap='RdYlGn',
                     s=8, alpha=0.5)
plt.colorbar(scatter, ax=ax, label='log(Prix)')
ax.set_title(f'Geo Distribution of Listings with GPS\n(n={len(gps):,})')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()


## 8. Correlation Analysis <a id='8'></a>

In [ ]:


# Columns to correlate
corr_cols = ['prix', 'surface', 'pieces', 'etage', 'annee_constr']
corr = df[corr_cols].corr()

# Figure
plt.figure(figsize=(8, 6))

# Mask upper triangle
mask = np.triu(np.ones_like(corr, dtype=bool))

# Heatmap
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    linewidths=0.5,
    square=True,
    cbar_kws={'shrink': 0.8}
)

plt.title('Correlation Matrix — Numeric Features', fontsize=14)
plt.show()

## 10. Data Quality Issues Summary <a id='10'></a>
Systematic review of all quality problems found above.

In [ ]:
issues = []

# Price
n = (df['prix'] <= 0).sum()
if n: issues.append(('prix ≤ 0',             n, 'Remove — impossible price'))
n = (df['prix'] > 50_000_000).sum()
if n: issues.append(('prix > 50M TND',        n, 'Remove — likely data entry error'))
issues.append(('prix missing',               df['prix'].isna().sum(), 'Impute or flag'))

# Surface
n = (df['surface'] <= 0).sum()
if n: issues.append(('surface ≤ 0',           n, 'Remove — impossible surface'))
n = (df['surface'] > 10_000).sum()
if n: issues.append(('surface > 10,000 m²',   n, 'Review — unusually large'))
issues.append(('surface missing',            df['surface'].isna().sum(), 'Impute or flag'))

# Construction year
n = (df['annee_constr'] < 1800).sum()
if n: issues.append(('annee_constr < 1800',   n, 'Set to NaN — impossible year'))
n = (df['annee_constr'] > datetime.now().year + 5).sum()
if n: issues.append(('annee_constr > now+5',  n, 'Set to NaN — future year'))

# Structural
issues.append(('code_postal 100% empty', len(df), 'DROP column entirely'))

# GPS
issues.append(('latitude/longitude sparse', df['latitude'].isna().sum(),
               f'Only {df["latitude"].notna().mean()*100:.1f}% have GPS — enrich via geocoding'))

issues_df = pd.DataFrame(issues, columns=['Issue', 'Count', 'Action'])
issues_df['Count'] = issues_df['Count'].apply(lambda x: f'{x:,}')
issues_df.style.set_properties(**{'text-align': 'left'})
issues_df


## 11. Data Preparation <a id='11'></a>
Applying all fixes identified above, plus feature engineering.

### 11.1  remove duplicates

In [ ]:
df_clean = dataset_full_raw.copy()

# Remove full duplicate rows
before = len(df_clean)
df_clean.reset_index(drop=True, inplace=True)
print(f'Exact duplicates removed : {before - len(df_clean):,}')

# Remove duplicate URLs (keep first occurrence)
if 'url' in df_clean.columns:
    before = len(df_clean)
    df_clean = df_clean[~df_clean['url'].duplicated(keep='first') | df_clean['url'].isna()]
    df_clean.reset_index(drop=True, inplace=True)
    print(f'Duplicate URLs removed   : {before - len(df_clean):,}')

print(f'\nShape after dedup: {df_clean.shape}')


### 11.2 Remove impossible / erroneous numeric values

In [ ]:
before = len(df_clean)

# Prix
df_clean = df_clean[~(df_clean['prix'].notna() & (df_clean['prix'] <= 0))]
df_clean = df_clean[~(df_clean['prix'].notna() & (df_clean['prix'] > 5_000_000))]

# Surface
df_clean = df_clean[~(df_clean['surface'].notna() & (df_clean['surface'] <= 0))]
df_clean = df_clean[~(df_clean['surface'].notna() & (df_clean['surface'] > 10_000))]

# Construction year — nullify out-of-range instead of dropping rows
df_clean.loc[df_clean['annee_constr'] < 1800, 'annee_constr'] = np.nan
df_clean.loc[df_clean['annee_constr'] > datetime.now().year + 5, 'annee_constr'] = np.nan

df_clean.reset_index(drop=True, inplace=True)
print(f'Rows removed for bad numeric values: {before - len(df_clean):,}')
print(f'Shape: {df_clean.shape}')


### 11.3 Standardize categorical text

In [ ]:
df_clean['contrat'] = df_clean['contrat'].str.strip().str.lower()
df_clean['type']    = df_clean['type'].str.strip().str.lower()
df_clean['gouvernerat'] = df_clean['gouvernerat'].str.strip().str.title()

# Normalize contrat variants
contrat_map = {
    'a vendre': 'vente', 'à vendre': 'vente',
    'a louer' : 'location', 'à louer': 'location',
    'location saisonnière': 'location_saisonniere',
}
df_clean['contrat'] = df_clean['contrat'].replace(contrat_map)

print('contrat value counts:')
print(df_clean['contrat'].value_counts().to_string())


### 11.4 Feature Engineering

In [ ]:
# Prix per m²
mask = df_clean['prix'].notna() & df_clean['surface'].notna() & (df_clean['surface'] > 0)
df_clean['prix_m2'] = np.nan
df_clean.loc[mask, 'prix_m2'] = (df_clean.loc[mask, 'prix'] / df_clean.loc[mask, 'surface']).round(0)
df_clean.loc[df_clean['prix_m2'] > 5000, 'prix_m2'] = np.nan  # cap extreme
df_clean.loc[df_clean['prix_m2'] < 10,     'prix_m2'] = np.nan

# Log transforms (for modeling — reduces skew)
df_clean['log_prix']    = np.log1p(df_clean['prix'])
df_clean['log_surface'] = np.log1p(df_clean['surface'])

# Date features
df_clean['date_publication'] = pd.to_datetime(df_clean['date_publication'], errors='coerce')
df_clean['pub_year']    = df_clean['date_publication'].dt.year.astype('Int64')
df_clean['pub_month']   = df_clean['date_publication'].dt.month.astype('Int64')
df_clean['pub_quarter'] = df_clean['date_publication'].dt.quarter.astype('Int64')

print('New columns created:')
new_cols = ['prix_m2', 'log_prix', 'log_surface', 'pub_year', 'pub_month', 'pub_quarter']
print(df_clean[new_cols].describe().round(2).T.to_string())


### 11.5 Recover missing `pieces` from text

In [ ]:
def extract_pieces(text):
    if pd.isna(text): return np.nan
    m = re.search(r'(\d+)\s*pi[eè]ces?', str(text).lower())
    if m: return float(m.group(1))
    m = re.search(r'(\d+)\s*ch(ambres?)?', str(text).lower())
    if m: return float(m.group(1))
    return np.nan

missing_pieces_mask = df_clean['pieces'].isna()
recovered = (
    df_clean.loc[missing_pieces_mask, 'description'].apply(extract_pieces)
    .combine_first(
    df_clean.loc[missing_pieces_mask, 'titre'].apply(extract_pieces))
)
df_clean.loc[missing_pieces_mask, 'pieces'] = recovered

n_recovered = (df_clean['pieces'].notna() & missing_pieces_mask).sum()
print(f'Pieces recovered from text: {n_recovered:,} rows')
print(f'Pieces coverage: {df_clean["pieces"].notna().mean()*100:.1f}%')


### 11.6 Binary proximity & amenity flags

In [ ]:
# Proximity columns → binary flags
proximity_cols = ['ecole','pharmacie','hopital','marche','magasin',
                  'bar','restaurant','transport']
for col in proximity_cols:
    if col in df_clean.columns:
        df_clean[f'prox_{col}'] = df_clean[col].notna().astype('int8')

# Amenity flags from caracteristiques
amenity_flags = ['piscine','parking','garage','jardin','terrasse',
                 'ascenseur','climatisation','balcon','gardien']
for amen in amenity_flags:
    df_clean[f'has_{amen}'] = df_clean['caracteristiques'].str.lower().str.contains(amen, na=False).astype('int8')

print('Proximity flags created:', [f'prox_{c}' for c in proximity_cols])
print('Amenity flags created  :', [f'has_{c}' for c in amenity_flags])


In [ ]:
# Clean distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col, color in zip(axes,
    ['log_prix', 'log_surface', 'prix_m2'],
    ['#3498db', '#2ecc71', '#e74c3c']):
    data = df_clean[col].dropna()
    ax.hist(data, bins=60, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(f'{col}  (n={len(data):,})')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')

plt.suptitle('Distributions After Cleaning & Feature Engineering',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Export
df_clean.to_csv('dataset_clean.csv', index=False)
print('✔  Exported → dataset_clean.csv')
print(f'   Shape: {df_clean.shape}')
print()
print('The clean DataFrame is available as df_clean')
df_clean.dtypes.to_frame('dtype')


# Stage 2

extraction des caracteristique par nlp 